In [1]:
import requests
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
load_dotenv()

True

In [37]:
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_community.utilities import DuckDuckGoSearchAPIWrapper

wrapper = DuckDuckGoSearchAPIWrapper(region='in-en')
search_tool = DuckDuckGoSearchRun()

In [53]:
@tool
def get_weather_data(city:str)->str:
    """
    This function fetches the current weather data for a given city
    """
    url = f'https://api.weatherstack.com/current?access_key=e2ab17c90d18f398e118cee53cf70a78&query={city}'
    response = requests.get(url).json()

    return response

In [54]:
llm = ChatOpenAI(model="gpt-6-luna")

In [55]:
from langchain.agents import create_agent
from langchain.agents.middleware import dynamic_prompt
from langsmith import Client
from langchain_core.prompts import SystemMessagePromptTemplate

In [56]:
client = Client()

In [57]:
prompt = client.pull_prompt("hwchase17/react", dangerously_pull_public_prompt=True)

In [58]:
@dynamic_prompt
def injection(request):


SyntaxError: incomplete input (1056270877.py, line 2)

In [59]:
agent = create_agent(
    model=llm,
    tools=[get_weather_data, search_tool],
    system_prompt="Answer the following questions as best you can.",
)

In [60]:
prompt

PromptTemplate(input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'], input_types={}, partial_variables={}, metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'}, template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}')

In [64]:
response = agent.invoke({
    "messages": [{"role": "user", "content": "Find the capital of Madhya Pradesh, then find Burhanpur's current weather condition"}]
})

In [62]:
from langchain_core.output_parsers import StrOutputParser
parser = StrOutputParser()

In [65]:
print(response['messages'][-1].text)

The capital of Madhya Pradesh is **Bhopal**. Burhanpur’s current weather is **clear**, around **21°C**.


In [66]:
print(response)

{'messages': [HumanMessage(content="Find the capital of Madhya Pradesh, then find Burhanpur's current weather condition", additional_kwargs={}, response_metadata={}, id='b8a2f0e5-e42b-447d-87e3-8f4a6244db7e'), AIMessage(content=[{'id': 'rs_0d4171fed536ecf9006b071c32946c87d0979d15188b9c5f31', 'summary': [], 'type': 'reasoning', 'content': [], 'encrypted_content': 'gAAAAABrBxwyRQYyYduKu-ebUDHbOwKbDqUqQ84D1EYuVZzLA_T2Ui5FMtU6sxe3T0248KS581r4Oob-6BVuKvEGtwafPlgAGLfyADUFD9B5_I7lgWYgHaA5qZd3v4a6U-Ea4OLmExianZmas_se_43Ye-Yg_YdVi9hUTiM7sYMW7HEud3o9gTm8EHJ-GzsqLJ6OgzC4WbUipyxze-WrhU5SGt9nh139W-b_BX-ASoUz2ctySRPJGAA91aYnLlJ4H21cjnY_nahMLnqkWAex3mo-s2Xr0fswe_4SlqlU0X8aeL6ImOSayesEkQ40xlmGZrZPXwKWM4kzs3h-FUXpq6rnwIXEMqXMwDrbbybAZ4V3fTOX71uO4UsVnNesMKRG7zqtTmV8Iwm6fjwZZ5TLiBjnFuF0zK_ZMGgIwEcNgs7g993GAJ8TBVW3nkizcA-gq_pGcfCwv-pN1R6bp4XeOzBHedC6bSr28IRhiuMSo46hsyifGiNlz6wTK_TjIf0PWTz9el1JxntGPpsz_TPsbdsEDPgKxJTwzZ1tlejr6g0DoXsznqtu7O0Ojcrr3WgTHbn4nroQRfGIM5a4HDZB7wzRnFyTx7-y9PiEQvbxcrIG8lS9Owtk1c91wz